# Screener.in document downloader (Google Colab)

Run the cells **in order**. Needs the folder `MyDrive/screener-document-download/` in your Google Drive with:
`code/`, `screener_auth.json` and the `equitydata-screener-all-stocks-*.csv` file.

1. **Cell 1** connects Drive and checks the files.
2. **Cell 2** holds the settings you may change.
3. **Cell 3** is a safe test (downloads nothing).
4. **Cell 4** is the real download. Change `START` in Cell 2 and re-run it for the next batch.
5. **Cell 5** shows progress. **Cell 6** syncs Drive when you finish.

Safe to stop and re-run: already downloaded files are skipped.

In [ ]:
# Cell 1 - connect Google Drive and check the project files
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/screener-document-download'  # change if your folder is elsewhere
%cd {PROJECT}

missing = [f for f in ('code/download_screener_docs.py', 'screener_auth.json') if not os.path.exists(f)]
csvs = sorted(f for f in os.listdir('.') if f.startswith('equitydata-screener-all-stocks-') and f.endswith('.csv'))
if missing or not csvs:
    raise FileNotFoundError(f'Missing files: {missing}; stock CSV found: {csvs}. Check the folder in Drive.')
print('OK - project found. Stock list:', csvs[-1])

In [ ]:
# Cell 2 - settings (edit these, then run the cell)
START = 0      # skip this many stocks from the CSV (next batch: 200, 400, ...)
LIMIT = 200    # how many stocks in this batch (5,541 stocks in total)
TYPES = ['annual', 'transcript', 'ppt']   # remove 'annual' to save Drive space
EXCEL = True   # also click 'Export to Excel' for every stock
WORKERS = 3    # stocks downloaded in parallel (1-8); each stock still runs start to finish on one worker

types_arg = '--types ' + ' '.join(TYPES)
excel_arg = '' if EXCEL else '--no-excel'
workers_arg = f'--workers {WORKERS}'
print(f'Batch: stocks {START} to {START + LIMIT - 1} | types: {TYPES} | excel: {EXCEL} | workers: {WORKERS}')

In [ ]:
# Cell 3 - TEST: 3 stocks, downloads nothing. Expect 'Found N candidate documents' and 'failed=0'.
# If you see a login/session error, export a fresh screener_auth.json and upload it again.
!python3 code/download_screener_docs.py --all --dry-run --start {START} --limit 3 {types_arg}

In [ ]:
# Cell 4 - REAL DOWNLOAD of this batch. Progress, counts and ETA print below.
# Files go to data/<SYMBOL>/ in your Drive; the log goes to logs/.
# If Colab disconnects: re-run Cells 1, 2 and 4 - finished files are skipped.
!python3 code/download_screener_docs.py --all --start {START} --limit {LIMIT} {types_arg} {excel_arg} {workers_arg}

In [ ]:
# Cell 5 - STATUS: totals of the last run + what is on disk
import glob

summaries = sorted(glob.glob('logs/run_*.summary.json'))
if summaries:
    print('Last run summary:', summaries[-1])
    print(open(summaries[-1]).read())

files = [os.path.join(r, f) for r, _, fs in os.walk('data') for f in fs] if os.path.isdir('data') else []
stocks = [d for d in os.listdir('data') if os.path.isdir(os.path.join('data', d)) and not d.startswith('_')] if files else []
size_gb = sum(os.path.getsize(f) for f in files) / 1024 ** 3
print(f'On Drive: {len(stocks)} stocks, {len(files)} files, {size_gb:.2f} GB (free Drive limit is 15 GB)')

logs = sorted(glob.glob('logs/run_*.log'))
if logs:
    print('\nLast log lines (' + logs[-1] + '):')
    print(''.join(open(logs[-1]).readlines()[-6:]))

In [ ]:
# Cell 6 - when finished: make sure everything is written to Drive
drive.flush_and_unmount()
print('Drive synced.')